# Notebook 4 — Recovery trajectories and candidate families

Twelve fixed study areas, two spatial supports, one sequence:
**spatial context → observed trajectories → analysis matrix → preprocessing → k-means diagnostics → recovery families → maps.**

The main signal is reliability-qualified NTL (`DNB_BRDF_Corrected_NTL`, MQF == 0). Municipality-wide means the GHSL G3 support within the whole boundary. Each local support is the same municipality's G3 pixels inside a 5×5 VIIRS window, anchored on its brightest pre-event baseline pixel. It can contain fewer than 25 eligible pixels. These are systematic anchors, not the named POIs in Notebook 3.

Run sequentially in the existing Black Marble environment. This revision is supplied with cleared outputs: the source datasets could not be accessed here, so no new empirical results are claimed. Existing project and export locations are retained. The two boundary-column settings below need confirmation against the local shapefile; they are explicit settings rather than a name-resolution framework.


In [8]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import geopandas as gpd
import xarray as xr
import rioxarray as rxr
from rasterio.enums import Resampling
from rasterio.features import rasterize
from shapely.geometry import box, Point
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from IPython.display import display


In [9]:
PROJECT_DIR_OVERRIDE = Path(
    "/Users/S4135723/Library/CloudStorage/OneDrive-RMITUniversity/"
    "02 - CH2 - Disaster Impact and Recovery/blackmarble-disaster-recovery"
)
PROJECT_DIR = PROJECT_DIR_OVERRIDE
DATA_DIR = PROJECT_DIR / "datasets"
VNP46_DIR = DATA_DIR / "VNP46"
PROCESSED_DIR = VNP46_DIR / "processed"
A2_ZARR_PATH = PROCESSED_DIR / "Haiyan_VNP46A2.zarr"
MUNICIPALITIES_PATH = DATA_DIR / "boundaries/MuniCities/MuniCities.shp"
HAIYAN_TRACK_PATH = DATA_DIR / "yolanda-path-line-/Yolanda Path Line.shp"
# Fixed filename; its directory was not recorded in the source notebook.
ghsl_paths = list(DATA_DIR.glob("**/GHSL_SMOD_E2015.tif"))
assert len(ghsl_paths) == 1, "Set GHSL_PATH to the existing GHSL_SMOD_E2015.tif."
GHSL_PATH = ghsl_paths[0]
NAME_COLUMN, PROVINCE_COLUMN = "NAME_2", "PROVINCE"

OUTPUT_DIR = PROJECT_DIR / "output/focused_recovery_walkthrough"
FIGURE_DIR, TABLE_DIR = OUTPUT_DIR / "figures", OUTPUT_DIR / "tables"
STUDY_AREAS = ["Tacloban", "Ormoc", "Baybay", "Catbalogan", "Borongan", "Guiuan",
               "Palo", "Tanauan", "Tolosa", "Dulag", "Basey", "Lawaan"]
study_area_provinces = dict(zip(STUDY_AREAS, [
    "Leyte", "Leyte", "Leyte", "Samar", "Eastern Samar", "Eastern Samar",
    "Leyte", "Leyte", "Leyte", "Leyte", "Samar", "Eastern Samar"
]))
EVENT_DATE = pd.Timestamp("2013-11-08")
BASELINE_DAYS, AGGREGATION_DAYS, KERNEL_SIZE = 60, 4, 5
BASELINE_START = EVENT_DATE - pd.Timedelta(days=BASELINE_DAYS)
PRE_EVENT_END = EVENT_DATE - pd.Timedelta(days=1)
ANALYSIS_START = EVENT_DATE - pd.Timedelta(days=180)
PROFILE_END = EVENT_DATE + pd.Timedelta(days=363)
DNB_BAND, MQF_BAND = "DNB_BRDF_Corrected_NTL", "Mandatory_Quality_Flag"
SPATIAL_DIMS = ("y", "x")
GHSL_MASKS, SETTLEMENT_MASK = {"G3": (22, 23, 30)}, "G3"
SPATIAL_COMPLETENESS_PCT = 10.0  # Original walkthrough setting; not an RQ1 optimum.
RQ_CLIP_PERCENTILE = 95.0
MIN_BASELINE_OBSERVATIONS = {4: 3}
CLUSTER_HORIZON_DAYS, CLUSTER_BIN_DAYS, MIN_BIN_COMPOSITES = 180, 20, 2
CLUSTER_RANDOM_STATE = 42
SUPPORTS = ["Municipality-wide G3", "Local 5×5 window"]


**Retained processing choices.** Four-day medians, a 60-day pixelwise baseline, at least three baseline composites per fixed pixel, and daily spatial P95 capping are unchanged. The original **10% spatial-completeness gate is permissive** and is retained for continuity, not presented as the RQ1 optimum. Family labels are conditional on this setting; review the coverage plots before interpreting them. Missing bins remain missing.


In [10]:
PLOT_FONT, PLOT_TEXT_COLOR = "Arial", "#243B5A"
EVENT_COLOR, BASELINE_COLOR = "#0057FF", "#6C7882"
AREA_COLORS = dict(zip(STUDY_AREAS, px.colors.qualitative.Dark24))
FAMILY_COLORS = ["#0072B2", "#E69F00", "#D55E00", "#009E73", "#CC79A7", "#56B4E9"]

def style_figure(fig, title):
    fig.update_layout(
        template="plotly_white", paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="white",
        width=1280, height=720, font=dict(family=PLOT_FONT, size=14, color=PLOT_TEXT_COLOR),
        title=dict(text=title, x=0.03, xanchor="left", font=dict(size=22)),
        margin=dict(l=75, r=45, t=90, b=115),
        legend=dict(orientation="h", x=0, y=-0.18, xanchor="left", yanchor="top"),
    )
    fig.update_xaxes(zeroline=False, gridcolor="#E8EDF3")
    fig.update_yaxes(zeroline=False, gridcolor="#E8EDF3")
    return fig

def finish_figure(fig, filename):
    FIGURE_DIR.mkdir(parents=True, exist_ok=True)
    fig.write_html(FIGURE_DIR / f"{filename}.html", include_plotlyjs="directory")
    fig.show()

def boundary_coordinates(geometry):
    boundary = geometry.boundary
    lines = list(boundary.geoms) if boundary.geom_type == "MultiLineString" else [boundary]
    x, y = [], []
    for line in lines:
        xx, yy = line.xy
        x.extend([*xx, None]); y.extend([*yy, None])
    return x, y

def add_haiyan_marker(fig, row=None, col=None):
    fig.add_vline(x=EVENT_DATE, line_dash="dash", line_color=EVENT_COLOR,
                  line_width=1.5, row=row, col=col)


## 1. Fix the spatial supports

Study-area and province matching is explicit. The short removal of “City of”/“City” below only translates boundary labels to the existing study names. It does not search for alternative columns or choose among ambiguous matches. A failed match stops here, before any recovery is calculated.


In [ ]:
municipalities = gpd.read_file(MUNICIPALITIES_PATH)
assert {NAME_COLUMN, PROVINCE_COLUMN}.issubset(municipalities.columns), (
    f"Set NAME_COLUMN and PROVINCE_COLUMN above. Available fields: {list(municipalities.columns)}"
)
municipalities["unit_name"] = (municipalities[NAME_COLUMN].str.upper()
    .str.replace("CITY OF ", "", regex=False).str.replace(" CITY", "", regex=False).str.strip().str.title())
municipalities["province_name"] = municipalities[PROVINCE_COLUMN].str.strip().str.title()
focused_municipalities = municipalities.loc[
    municipalities.unit_name.isin(STUDY_AREAS)
    & municipalities.province_name.eq(municipalities.unit_name.map(study_area_provinces))
].copy()
assert focused_municipalities.unit_name.value_counts().reindex(STUDY_AREAS, fill_value=0).eq(1).all(), (
    "Expected one boundary per study area. Check the explicit names and province labels."
)
focused_municipalities = focused_municipalities.set_index("unit_name").loc[STUDY_AREAS].reset_index()
focused_municipalities["profile_id"] = np.arange(1, len(STUDY_AREAS) + 1)
municipalities_display = focused_municipalities.to_crs("EPSG:4326")
regional_boundaries = municipalities.to_crs("EPSG:4326")
haiyan_track = gpd.read_file(HAIYAN_TRACK_PATH).to_crs("EPSG:4326")

# The saved source run identifies this VIIRS grid as EPSG:4326.
a2 = xr.open_zarr(A2_ZARR_PATH, consolidated=None, chunks="auto", mask_and_scale=True)
a2 = a2.set_coords("date").swap_dims({a2.date.dims[0]: "date"}).sortby("date")
a2 = a2.assign_coords(date=pd.to_datetime(a2.date.values).normalize())
assert a2.indexes["date"].is_unique, "Resolve duplicate observation dates before compositing."
a2 = a2.sel(date=slice(ANALYSIS_START, PROFILE_END)).rio.write_crs("EPSG:4326")
dnb = a2[DNB_BAND].astype("float32")
dnb = dnb.where(np.isfinite(dnb) & (dnb >= 0) & ~dnb.isin([6553.5, 65535.0]))
mqf = a2[MQF_BAND]
ghsl = rxr.open_rasterio(GHSL_PATH, masked=True).isel(band=0, drop=True)
ghsl_viirs = ghsl.rio.reproject_match(dnb.isel(date=0), resampling=Resampling.nearest)
ghsl_viirs = ghsl_viirs.assign_coords(x=dnb.x, y=dnb.y)
ghsl_mask = ghsl_viirs.isin(GHSL_MASKS[SETTLEMENT_MASK])
municipalities_raster_crs = focused_municipalities.to_crs(a2.rio.crs)
all_zone_id = xr.DataArray(rasterize(
    list(zip(municipalities_raster_crs.geometry, municipalities_raster_crs.profile_id)),
    out_shape=(dnb.sizes["y"], dnb.sizes["x"]), transform=dnb.rio.transform(),
    fill=0, all_touched=False, dtype="int32"
), dims=SPATIAL_DIMS, coords={"y": dnb.y, "x": dnb.x})
rq_base_mask = ghsl_mask & (all_zone_id > 0)
rq_unclipped = dnb.where((mqf == 0) & rq_base_mask)
rq_daily_p95 = rq_unclipped.chunk({"y": -1, "x": -1}).quantile(
    RQ_CLIP_PERCENTILE / 100, dim=SPATIAL_DIMS, skipna=True).compute()
rq_cube = xr.where(rq_unclipped > rq_daily_p95, rq_daily_p95, rq_unclipped)


In [ ]:
def build_pixel_matched_profile(cube, support_mask, unit_name, unit_type):
    # Reindex every calendar block: wholly absent periods must remain visible.
    blocks = np.arange((ANALYSIS_START - EVENT_DATE).days // AGGREGATION_DAYS,
                       (PROFILE_END - EVENT_DATE).days // AGGREGATION_DAYS + 1)
    day_blocks = (pd.DatetimeIndex(cube.date.values) - EVENT_DATE).days // AGGREGATION_DAYS
    composites = (cube.where(support_mask).assign_coords(block=("date", day_blocks))
        .groupby("block").median("date", skipna=True).reindex(block=blocks).compute())
    baseline_blocks = composites.sel(block=slice(-BASELINE_DAYS // AGGREGATION_DAYS, -1))
    ntl0 = baseline_blocks.median("block", skipna=True)
    fixed_mask = (support_mask & (baseline_blocks.count("block") >= MIN_BASELINE_OBSERVATIONS[4])
                  & np.isfinite(ntl0) & (ntl0 > 0))
    fixed_pixel_count = int(fixed_mask.sum())
    paired_valid = composites.notnull() & fixed_mask
    valid_pixel_count = paired_valid.sum(SPATIAL_DIMS)
    spatial_coverage_pct = 100 * valid_pixel_count / max(fixed_pixel_count, 1)
    current_radiance = composites.where(paired_valid).sum(SPATIAL_DIMS, min_count=1)
    matched_baseline = ntl0.where(paired_valid).sum(SPATIAL_DIMS, min_count=1)
    reduced = xr.Dataset({
        "recovery_pct": 100 * current_radiance / matched_baseline,
        "current_radiance": current_radiance,
        "matched_baseline_radiance": matched_baseline,
        "raw_median_ntl": composites.where(paired_valid).median(SPATIAL_DIMS, skipna=True),
        "spatial_coverage_pct": spatial_coverage_pct,
        "valid_pixel_count": valid_pixel_count,
    })
    profile = reduced.to_dataframe().reset_index()
    profile["date_start"] = EVENT_DATE + pd.to_timedelta(profile.block * AGGREGATION_DAYS, unit="D")
    profile["date_end"] = profile.date_start + pd.Timedelta(days=AGGREGATION_DAYS - 1)
    profile["date_mid"] = profile.date_start + pd.Timedelta(days=1.5)
    withheld = profile.spatial_coverage_pct.lt(SPATIAL_COMPLETENESS_PCT)
    profile.loc[withheld, ["recovery_pct", "current_radiance", "matched_baseline_radiance", "raw_median_ntl"]] = np.nan
    profile["observation_status"] = np.where(profile.recovery_pct.notna(), "observed", "not observable")
    profile["unit_name"], profile["unit_type"] = unit_name, unit_type
    profile["fixed_baseline_pixels"] = fixed_pixel_count
    profile["ghsl_mask"], profile["sc_threshold_pct"] = SETTLEMENT_MASK, SPATIAL_COMPLETENESS_PCT
    profile["baseline_start"], profile["baseline_end"] = BASELINE_START, PRE_EVENT_END
    profile["aggregation_days"] = AGGREGATION_DAYS
    baseline_values = ntl0.where(fixed_mask).values
    report = dict(unit_name=unit_name, unit_type=unit_type, fixed_baseline_pixels=fixed_pixel_count,
                  baseline_median_ntl=float(np.nanmedian(baseline_values)) if fixed_pixel_count else np.nan)
    return profile, composites.where(fixed_mask), ntl0.where(fixed_mask), fixed_mask, report

municipality_profiles, kernel_profiles = [], []
municipality_reports, kernel_reports, kernel_anchors = [], [], []
baseline_surfaces, fixed_masks, municipality_composites = {}, {}, {}
kernel_windows = {}
for row in municipalities_raster_crs.itertuples():
    support = (all_zone_id == row.profile_id) & ghsl_mask
    yy, xx = np.where(support.values)
    assert len(yy), f"{row.unit_name}: no G3 support on the VIIRS grid."
    sy, sx = slice(yy.min(), yy.max()+1), slice(xx.min(), xx.max()+1)
    profile, composites, baseline, fixed, report = build_pixel_matched_profile(
        rq_cube.isel(y=sy, x=sx), support.isel(y=sy, x=sx), row.unit_name, SUPPORTS[0])
    profile["profile_id"] = row.profile_id
    municipality_profiles.append(profile); municipality_reports.append(report)
    baseline_surfaces[row.profile_id], fixed_masks[row.profile_id] = baseline, fixed
    municipality_composites[row.unit_name] = composites
    if not report["fixed_baseline_pixels"]:
        continue  # This area remains in the eligibility table as unavailable.
    iy, ix = np.unravel_index(np.nanargmax(baseline.values), baseline.shape)
    anchor_x, anchor_y = float(baseline.x[ix]), float(baseline.y[iy])
    gx, gy = int(np.abs(dnb.x.values-anchor_x).argmin()), int(np.abs(dnb.y.values-anchor_y).argmin())
    assert 2 <= gx < dnb.sizes["x"]-2 and 2 <= gy < dnb.sizes["y"]-2, "Anchor too close to raster edge."
    sy, sx = slice(gy-2, gy+3), slice(gx-2, gx+3)
    profile, _, _, _, report = build_pixel_matched_profile(
        rq_cube.isel(y=sy, x=sx), support.isel(y=sy, x=sx), row.unit_name, SUPPORTS[1])
    profile["profile_id"] = row.profile_id
    kernel_profiles.append(profile); kernel_reports.append(report)
    dx, dy = abs(float(dnb.x[1]-dnb.x[0])), abs(float(dnb.y[1]-dnb.y[0]))
    kernel_windows[row.unit_name] = box(anchor_x-2.5*dx, anchor_y-2.5*dy,
                                         anchor_x+2.5*dx, anchor_y+2.5*dy)
    kernel_anchors.append(dict(profile_id=row.profile_id, unit_name=row.unit_name,
        anchor_x=anchor_x, anchor_y=anchor_y, anchor_baseline_ntl=float(baseline.values[iy, ix]),
        g3_municipal_pixels_in_window=int(support.isel(y=sy, x=sx).sum())))
municipality_four_day = pd.concat(municipality_profiles, ignore_index=True)
kernel_four_day = (pd.concat(kernel_profiles, ignore_index=True) if kernel_profiles
                   else municipality_four_day.iloc[:0].copy())
municipality_reports = pd.DataFrame(municipality_reports)
kernel_reports = pd.DataFrame(kernel_reports, columns=municipality_reports.columns)
kernel_anchors = pd.DataFrame(kernel_anchors, columns=["profile_id", "unit_name", "anchor_x", "anchor_y",
                                                     "anchor_baseline_ntl", "g3_municipal_pixels_in_window"])


In [ ]:
# The same extent and contextual layers are reused for the family maps.
x0, y0, x1, y1 = municipalities_display.total_bounds
map_bounds = [x0-0.12, y0-0.12, x1+0.12, y1+0.12]
regional_display = regional_boundaries.cx[map_bounds[0]:map_bounds[2], map_bounds[1]:map_bounds[3]]
context_x, context_y = [], []
for geometry in regional_display.geometry:
    xx, yy = boundary_coordinates(geometry)
    context_x.extend(xx); context_y.extend(yy)
track_x, track_y = [], []
for geometry in haiyan_track.geometry:
    lines = list(geometry.geoms) if geometry.geom_type == "MultiLineString" else [geometry]
    for line in lines:
        xx, yy = line.xy
        track_x.extend([*xx, None]); track_y.extend([*yy, None])

def map_context(fig, row, col):
    fig.add_trace(go.Scatter(x=context_x, y=context_y, mode="lines", line=dict(color="#CFD6DE", width=0.7),
                            showlegend=False, hoverinfo="skip"), row=row, col=col)
    for color, width in [("white", 5), ("#111111", 2)]:
        fig.add_trace(go.Scatter(x=track_x, y=track_y, mode="lines", line=dict(color=color, width=width),
                                showlegend=False, hoverinfo="skip"), row=row, col=col)
    fig.update_xaxes(range=[map_bounds[0], map_bounds[2]], ticksuffix="°E", title_text="Longitude", row=row, col=col)
    fig.update_yaxes(range=[map_bounds[1], map_bounds[3]], ticksuffix="°N", title_text="Latitude", row=row, col=col)
    axis_number = (row-1)*2+col
    fig.update_yaxes(scaleanchor="x" if axis_number == 1 else f"x{axis_number}",
                     scaleratio=1/np.cos(np.deg2rad((y0+y1)/2)), row=row, col=col)

fig = make_subplots(rows=1, cols=2, subplot_titles=["Whole-municipality boundaries", "5×5 windows and anchors"])
for col in (1, 2):
    map_context(fig, 1, col)
    for row in municipalities_display.itertuples():
        color = AREA_COLORS[row.unit_name]
        xx, yy = boundary_coordinates(row.geometry)
        fig.add_trace(go.Scatter(x=xx, y=yy, mode="lines", line=dict(color=color, width=2 if col==1 else 0.8),
                                showlegend=False, hoverinfo="skip"), row=1, col=col)
        point = row.geometry.representative_point()
        if col == 2 and row.unit_name in kernel_windows:
            xx, yy = boundary_coordinates(kernel_windows[row.unit_name])
            fig.add_trace(go.Scatter(x=xx, y=yy, mode="lines", line=dict(color=color, width=2),
                                    showlegend=False, hoverinfo="skip"), row=1, col=col)
            anchor = kernel_anchors.set_index("unit_name").loc[row.unit_name]
            point = Point(anchor.anchor_x, anchor.anchor_y)
        fig.add_trace(go.Scatter(x=[point.x], y=[point.y], mode="markers+text", text=[row.unit_name],
            textposition="top center", textfont=dict(size=10), marker=dict(color=color, size=6),
            showlegend=False, hovertemplate=row.unit_name+"<extra></extra>"), row=1, col=col)
style_figure(fig, "Fixed study areas and spatial supports · black line: Haiyan path")
finish_figure(fig, "01_study_areas_and_5x5")

support_counts = pd.concat([municipality_reports, kernel_reports], ignore_index=True)
fig = px.bar(support_counts, x="unit_name", y="fixed_baseline_pixels", color="unit_type", barmode="group",
             color_discrete_sequence=["#174A7E", "#E69F00"],
             labels={"fixed_baseline_pixels":"Fixed baseline-lit G3 pixels", "unit_name":"Study area", "unit_type":"Support"})
fig.update_yaxes(type="log")
style_figure(fig, "How many pixels support each trajectory? · logarithmic pixel count")
finish_figure(fig, "02_fixed_support_counts")


## 2. Observe the trajectories before summarising them

The paired support plots use the same baseline definition and completeness gate. Lines stop at missing four-day blocks. Coverage strips show the fraction of the fixed baseline-lit G3 support observed in each composite; they measure observability, not recovery. The horizontal line is 100% of the matched baseline. The vertical dashed blue line marks Haiyan on 8 November 2013.


In [ ]:
for support_name, profiles, slug in [(SUPPORTS[0], municipality_four_day, "municipality"),
                                     (SUPPORTS[1], kernel_four_day, "5x5")]:
    fig = make_subplots(rows=4, cols=3, subplot_titles=STUDY_AREAS,
                        shared_xaxes=True, vertical_spacing=0.10)
    for i, name in enumerate(STUDY_AREAS):
        r, c = i//3+1, i%3+1
        group = profiles.loc[profiles.unit_name.eq(name)]
        fig.add_trace(go.Scatter(x=group.date_mid, y=group.recovery_pct, mode="lines+markers",
            connectgaps=False, line=dict(color=AREA_COLORS[name], width=1.5), marker=dict(size=3),
            customdata=group[["spatial_coverage_pct", "valid_pixel_count"]], showlegend=False,
            hovertemplate="%{x|%d %b %Y}<br>%{y:.1f}% baseline<br>Coverage %{customdata[0]:.1f}%"
                          "<br>Valid pixels %{customdata[1]}<extra></extra>"), row=r, col=c)
        add_haiyan_marker(fig, r, c)
        fig.add_hline(y=100, line_color=BASELINE_COLOR, line_dash="dot", row=r, col=c)
        fig.update_xaxes(range=[BASELINE_START, PROFILE_END], tickformat="%b %Y", row=r, col=c)
        fig.update_yaxes(title_text="% baseline" if c==1 else None, row=r, col=c)
    style_figure(fig, f"{support_name} · observed four-day recovery trajectories")
    fig.update_layout(height=960)
    finish_figure(fig, f"03_{slug}_trajectories")
    coverage = profiles.pivot(index="unit_name", columns="date_mid", values="spatial_coverage_pct").reindex(STUDY_AREAS)
    fig = go.Figure(go.Heatmap(x=coverage.columns, y=coverage.index, z=coverage.values,
        zmin=0, zmax=100, colorscale="Greens", colorbar=dict(title="Coverage %"),
        hovertemplate="%{y}<br>%{x|%d %b %Y}<br>Observed fixed pixels %{z:.1f}%<extra></extra>"))
    add_haiyan_marker(fig)
    style_figure(fig, f"{support_name} · observability behind the trajectories")
    fig.update_xaxes(range=[BASELINE_START, PROFILE_END])
    finish_figure(fig, f"04_{slug}_coverage")


In [ ]:
# A compact check of absolute radiance and the influence of baseline brightness.
brightness_rows = []
for support_name, profiles, reports in [(SUPPORTS[0], municipality_four_day, municipality_reports),
                                        (SUPPORTS[1], kernel_four_day, kernel_reports)]:
    for name, group in profiles.groupby("unit_name", sort=False):
        pre = group.loc[group.date_start.between(BASELINE_START, PRE_EVENT_END), "recovery_pct"].dropna()
        report = reports.set_index("unit_name").loc[name]
        brightness_rows.append(dict(unit_name=name, unit_type=support_name,
            baseline_median_ntl=report.baseline_median_ntl,
            fixed_baseline_pixels=report.fixed_baseline_pixels,
            pre_event_mad_pct=(pre-pre.median()).abs().median()))
brightness_diagnostic = pd.DataFrame(brightness_rows)
fig = px.scatter(brightness_diagnostic, x="baseline_median_ntl", y="pre_event_mad_pct",
    color="unit_type", symbol="unit_type", hover_name="unit_name",
    hover_data=["fixed_baseline_pixels"], color_discrete_sequence=["#174A7E", "#E69F00"],
    labels={"baseline_median_ntl":"Baseline radiance (nW cm⁻² sr⁻¹)",
            "pre_event_mad_pct":"Pre-event MAD (percentage points)", "unit_type":"Support"})
style_figure(fig, "Check dim or volatile baselines before clustering")
finish_figure(fig, "05_baseline_variability")


## 3. Compile the recovery trajectories

Each row is one study area at one spatial support. Each column is a non-overlapping 20-day interval during days 0–179 after Haiyan. A feature is the median of at least two admissible four-day composites. Counts accompany every value; blanks mean insufficient observation, not zero recovery.

Both supports retain the existing matched-baseline percentages. These features describe NTL relative to baseline, not the fraction of disaster damage repaired.


In [ ]:
trajectories = pd.concat([municipality_four_day, kernel_four_day], ignore_index=True)
support_order = ["Municipality-wide G3", "Local 5×5 window"]
profile_index = pd.MultiIndex.from_product([support_order, STUDY_AREAS], names=["unit_type", "unit_name"])
number_of_bins = CLUSTER_HORIZON_DAYS // CLUSTER_BIN_DAYS
feature_labels = [f"{i * CLUSTER_BIN_DAYS}–{(i + 1) * CLUSTER_BIN_DAYS - 1} d" for i in range(number_of_bins)]
feature_days = np.arange(number_of_bins) * CLUSTER_BIN_DAYS + (CLUSTER_BIN_DAYS - 1) / 2
selected = trajectories.loc[trajectories.block.ge(0) & (trajectories.block * AGGREGATION_DAYS).lt(CLUSTER_HORIZON_DAYS)].copy()
selected["cluster_bin"] = selected.block * AGGREGATION_DAYS // CLUSTER_BIN_DAYS
grouped = selected.groupby(["unit_type", "unit_name", "cluster_bin"], observed=True).recovery_pct
clustering_counts = grouped.count().unstack("cluster_bin").reindex(index=profile_index, columns=range(number_of_bins)).fillna(0).astype(int)
clustering_features = grouped.median().unstack("cluster_bin").reindex(index=profile_index, columns=range(number_of_bins))
clustering_features = clustering_features.where(clustering_counts >= MIN_BIN_COMPOSITES)
clustering_features.columns = clustering_counts.columns = feature_labels
municipality_features = clustering_features.xs(support_order[0])
municipality_counts = clustering_counts.xs(support_order[0])

for support, prefix in zip(support_order, ["municipality", "5x5"]):
    values, counts = clustering_features.xs(support), clustering_counts.xs(support)
    fig = make_subplots(rows=1, cols=2, subplot_titles=("Baseline-relative NTL: 20-day medians", "Admissible four-day composites"), horizontal_spacing=0.16)
    fig.add_trace(go.Heatmap(x=feature_labels, y=values.index, z=values.values, colorscale="Viridis", colorbar=dict(title="% baseline", x=0.43, thickness=12), hoverongaps=False, hovertemplate="%{y}<br>%{x}: %{z:.1f}%<extra></extra>"), row=1, col=1)
    fig.add_trace(go.Heatmap(x=feature_labels, y=counts.index, z=counts.values, colorscale="Blues", zmin=0, zmax=CLUSTER_BIN_DAYS // AGGREGATION_DAYS, colorbar=dict(title="Count", thickness=12), hovertemplate="%{y}<br>%{x}: %{z} composites<extra></extra>"), row=1, col=2)
    fig.update_yaxes(autorange="reversed")
    fig.update_xaxes(tickangle=-45)
    style_figure(fig, f"Clustering evidence · {support}")
    finish_figure(fig, f"clustering_matrix_{prefix}")


## 4–5. Preprocess transparently and inspect candidate k

Clustering uses complete rows only: all nine intervals must meet the observation-count rule. No interpolation, extrapolation, zero filling, or within-trajectory standardisation is applied. Equal-duration features share the same percentage scale, so amplitude and persistent deficits remain part of the distance. The eligibility table records every retained and excluded support; complete cases may represent the more observable locations.

Fit the two supports separately. Inspect inertia (the elbow) and mean silhouette together. The elbow is the largest downward departure from the chord joining the first and last normalised inertia values; fewer than three candidate k values cannot identify an elbow. Select the highest silhouette among partitions without singletons when available; otherwise retain the highest finite score as an explicitly exploratory solution. A disagreement between diagnostics remains visible. Neither diagnostic establishes a unique or externally validated optimum.


In [ ]:
eligibility = pd.DataFrame(index=profile_index)
eligibility["available_bins"] = clustering_features.notna().sum(axis=1)
eligibility["required_bins"] = number_of_bins
eligibility["retained_composites"] = clustering_counts.sum(axis=1)
eligibility["eligible"] = eligibility.available_bins.eq(number_of_bins)
eligibility["exclusion_reason"] = ["" if n == number_of_bins else f"{number_of_bins - n} bins below {MIN_BIN_COMPOSITES} admissible composites" for n in eligibility.available_bins]
complete_features_all = clustering_features.loc[eligibility.eligible]
complete_features = municipality_features.dropna()
assignments = eligibility.reset_index()
assignments["family_id"] = np.nan
assignments["family"] = "Not grouped"
assignments["clustering_status"] = "Insufficient observed bins"
clustering_models, diagnostic_rows = {}, []

for support, prefix in zip(support_order, ["M", "K"]):
    features = clustering_features.xs(support).dropna()
    X = features.to_numpy(float)
    n_unique = len(np.unique(X, axis=0))
    result = {"chosen_k": None, "elbow_k": np.nan, "model": None, "features": features}
    clustering_models[support] = result
    if len(X) < 3 or n_unique < 2:
        result["status"] = "Not fitted: fewer than 3 complete profiles or fewer than 2 distinct trajectories"
        mask = assignments.unit_type.eq(support) & assignments.eligible
        assignments.loc[mask, ["clustering_status", "exclusion_reason"]] = result["status"]
        continue
    candidates, rows = {}, []
    for k in range(1, min(6, len(X) - 1, n_unique) + 1):
        fitted = KMeans(n_clusters=k, n_init=20, random_state=CLUSTER_RANDOM_STATE).fit(X)
        labels = fitted.labels_
        candidates[k] = fitted
        rows.append({"unit_type": support, "k": k, "inertia": fitted.inertia_, "silhouette": silhouette_score(X, labels) if 1 < len(np.unique(labels)) < len(X) else np.nan, "minimum_family_size": int(np.bincount(labels).min())})
    diagnostic = pd.DataFrame(rows)
    elbow_k = np.nan
    if len(diagnostic) >= 3 and diagnostic.inertia.iloc[0] > diagnostic.inertia.iloc[-1]:
        x = (diagnostic.k - diagnostic.k.iloc[0]) / (diagnostic.k.iloc[-1] - diagnostic.k.iloc[0])
        y = (diagnostic.inertia - diagnostic.inertia.iloc[-1]) / (diagnostic.inertia.iloc[0] - diagnostic.inertia.iloc[-1])
        distance = (1 - x) - y
        if distance.iloc[1:-1].max() > 1e-9:
            elbow_k = int(diagnostic.loc[distance.iloc[1:-1].idxmax(), "k"])
    scored = diagnostic.loc[diagnostic.silhouette.notna()]
    non_singleton = scored.loc[scored.minimum_family_size.ge(2)]
    selection = non_singleton if len(non_singleton) else scored
    selected_k = int(selection.sort_values(["silhouette", "k"], ascending=[False, True]).iloc[0].k)
    fitted = candidates[selected_k]
    order = np.argsort(fitted.cluster_centers_.mean(axis=1), kind="stable")
    label_map = {int(old): new for new, old in enumerate(order)}
    labels = np.array([label_map[int(label)] for label in fitted.labels_])
    status = "Exploratory; singleton family present" if non_singleton.empty else "Exploratory; no singleton families"
    status += "; elbow unavailable" if pd.isna(elbow_k) else ("; diagnostics agree" if elbow_k == selected_k else f"; elbow suggests k={int(elbow_k)}")
    result.update(chosen_k=selected_k, elbow_k=elbow_k, model=fitted, labels=labels, status=status)
    diagnostic["chosen_k"], diagnostic["elbow_k"] = selected_k, elbow_k
    diagnostic_rows.extend(diagnostic.to_dict("records"))
    for name, label in zip(features.index, labels):
        mask = assignments.unit_type.eq(support) & assignments.unit_name.eq(name)
        assignments.loc[mask, ["family_id", "family", "clustering_status"]] = [label, f"{prefix}{label + 1}", status]

diagnostics_all = pd.DataFrame(diagnostic_rows, columns=["unit_type", "k", "inertia", "silhouette", "minimum_family_size", "chosen_k", "elbow_k"])
family_id = assignments.loc[assignments.unit_type.eq(support_order[0])].set_index("unit_name").family_id
chosen_k = clustering_models[support_order[0]]["chosen_k"]
display(assignments[["unit_type", "unit_name", "available_bins", "eligible", "family", "exclusion_reason"]])


In [ ]:
fig = make_subplots(rows=2, cols=2, subplot_titles=[f"{support} · {metric}" for support in support_order for metric in ["Inertia / elbow", "Mean silhouette"]], vertical_spacing=0.18)
for row, support in enumerate(support_order, start=1):
    diagnostic = diagnostics_all.loc[diagnostics_all.unit_type.eq(support)]
    result = clustering_models[support]
    print(f"{support}: {result['status']}")
    if diagnostic.empty:
        fig.add_annotation(x=0.5, y=0.5, xref=f"x{2 * row - 1 if row > 1 else ''} domain", yref=f"y{2 * row - 1 if row > 1 else ''} domain", text="Insufficient complete trajectories", showarrow=False)
        continue
    for column, metric in [(1, "inertia"), (2, "silhouette")]:
        fig.add_trace(go.Scatter(x=diagnostic.k, y=diagnostic[metric], mode="lines+markers", line=dict(color="#174A7E"), showlegend=False, customdata=diagnostic[["minimum_family_size"]], hovertemplate="k=%{x}<br>Value: %{y:.3f}<br>Smallest family: %{customdata[0]}<extra></extra>"), row=row, col=column)
        fig.add_vline(x=result["chosen_k"], line_dash="dash", line_color="#D55E00", row=row, col=column)
        fig.update_xaxes(title_text="Number of families (k)", dtick=1, row=row, col=column)
    if pd.notna(result["elbow_k"]):
        fig.add_vline(x=result["elbow_k"], line_dash="dot", line_color="#009E73", row=row, col=1)
    fig.add_hline(y=0, line_width=1, line_color="#BBBBBB", row=row, col=2)
style_figure(fig, "Candidate k · orange dashed = selected; green dotted = elbow")
finish_figure(fig, "clustering_k_diagnostics")


## 6. Describe the candidate trajectory families

Thin curves retain individual trajectories; thick dashed curves show k-means means. The companion panels show pointwise medians and interquartile envelopes across the same members. These are functional-style descriptive summaries, **not formal depth-based functional boxplots or uncertainty intervals**. A singleton has no between-member variability to summarise.

M1, M2, … identify municipality families; K1, K2, … identify local 5×5 families. Within each support, numbers run from lower to higher mean baseline-relative NTL. The independently fitted families are support-specific; matching numbers or colours do not establish equivalent membership or recovery behaviour.


In [ ]:
fig = make_subplots(rows=2, cols=2, subplot_titles=[f"{support} · {view}" for support in support_order for view in ["Members and mean", "Pointwise median and IQR"]], shared_xaxes=True, vertical_spacing=0.18)
summary_rows = []
for row, support in enumerate(support_order, start=1):
    result = clustering_models[support]
    if result["chosen_k"] is None:
        continue
    features = result["features"]
    for family in range(result["chosen_k"]):
        members = assignments.loc[assignments.unit_type.eq(support) & assignments.family_id.eq(family)]
        label, colour = members.family.iloc[0], FAMILY_COLORS[family]
        values = features.loc[members.unit_name]
        mean = values.mean(axis=0)
        q25, median, q75 = values.quantile([0.25, 0.5, 0.75]).to_numpy()
        for name, curve in values.iterrows():
            fig.add_trace(go.Scatter(x=feature_days, y=curve, mode="lines+markers", line=dict(color=colour, width=1), marker=dict(size=3), opacity=0.45, name=f"{label} · {name}", legendgroup=label, showlegend=False), row=row, col=1)
        fig.add_trace(go.Scatter(x=feature_days, y=mean, mode="lines+markers", line=dict(color=colour, width=3, dash="dash"), name=f"{label} (n={len(values)})", legendgroup=label), row=row, col=1)
        fig.add_trace(go.Scatter(x=feature_days, y=q25, mode="lines", line=dict(width=0), showlegend=False, hoverinfo="skip", legendgroup=label), row=row, col=2)
        fig.add_trace(go.Scatter(x=feature_days, y=q75, mode="lines", line=dict(width=0), fill="tonexty", fillcolor=colour, opacity=0.16, showlegend=False, hoverinfo="skip", legendgroup=label), row=row, col=2)
        fig.add_trace(go.Scatter(x=feature_days, y=median, mode="lines+markers", line=dict(color=colour, width=3), name=f"{label} median", showlegend=False, legendgroup=label), row=row, col=2)
        for i, feature in enumerate(feature_labels):
            summary_rows.append({"unit_type": support, "family_id": family, "family": label, "n_members": len(values), "members": ", ".join(values.index), "interval": feature, "mean_pct": mean.iloc[i], "median_pct": median[i], "q25_pct": q25[i], "q75_pct": q75[i]})
    for column in (1, 2):
        fig.add_hline(y=100, line_dash="dot", line_color="#808080", row=row, col=column)
        fig.add_vline(x=0, line_dash="dash", line_color="#0057FF", row=row, col=column)
        fig.update_yaxes(title_text="NTL (% matched baseline)", row=row, col=column)
        fig.update_xaxes(title_text="Days since Haiyan", row=row, col=column)
family_summaries = pd.DataFrame(summary_rows, columns=["unit_type", "family_id", "family", "n_members", "members", "interval", "mean_pct", "median_pct", "q25_pct", "q75_pct"])
style_figure(fig, "Candidate families · observed-bin trajectories and descriptive envelopes")
finish_figure(fig, "clustering_family_trajectories")
display(assignments[["unit_type", "unit_name", "family", "clustering_status", "exclusion_reason"]])


## 7. Project the families back to space

Municipality families colour the boundary; local families mark the 5×5 anchor and its window. Grey means **not grouped**, with the reason available on hover. M and K labels belong to separate fitted models: the same colour across panels does not imply the same recovery pattern. These maps show numerical trajectory similarity, not disaster causation or validated electricity restoration.


In [ ]:
fig = make_subplots(rows=1, cols=2, subplot_titles=SUPPORTS)
for col, support_name in enumerate(SUPPORTS, 1):
    map_context(fig, 1, col)
    lookup = assignments.loc[assignments.unit_type.eq(support_name)].set_index("unit_name")
    seen = set()
    for row in municipalities_display.itertuples():
        result = lookup.loc[row.unit_name]
        grouped = pd.notna(result.family_id)
        color = FAMILY_COLORS[int(result.family_id)] if grouped else "#9BA3AC"
        label = result.family if grouped else "Not grouped"
        geometry = row.geometry if col == 1 else kernel_windows.get(row.unit_name, row.geometry)
        xx, yy = boundary_coordinates(geometry)
        fig.add_trace(go.Scatter(x=xx, y=yy, mode="lines", line=dict(color=color, width=2.5),
            showlegend=False, hoverinfo="skip"), row=1, col=col)
        point = geometry.representative_point()
        fig.add_trace(go.Scatter(x=[point.x], y=[point.y], mode="markers+text", text=[row.unit_name],
            textposition="top center", textfont=dict(size=10), marker=dict(color=color, size=9, line=dict(color="white", width=1)),
            name=label, legendgroup=f"{col}-{label}", showlegend=label not in seen,
            hovertemplate=f"{row.unit_name}<br>{label}<br>{result.exclusion_reason}<extra></extra>"), row=1, col=col)
        seen.add(label)
style_figure(fig, "Candidate recovery families · municipality and local supports")
finish_figure(fig, "09_recovery_family_maps")


## Export and next step

Exports retain the original profile, anchor, feature, family and baseline-diagnostic filenames where their meaning is unchanged. Additional tables retain both supports, bin counts, excluded trajectories and k-selection diagnostics. The settings file records the signal, baseline, spatial threshold, GHSL class and event window. No missing observations are filled, and excluded locations remain visible.

**Next stage:** compare the observed families and their quality flags with physical and life-quality layers. That comparison is intentionally outside this notebook.


In [ ]:
TABLE_DIR.mkdir(parents=True, exist_ok=True)
municipality_four_day.to_csv(TABLE_DIR / "focused_municipality_four_day_profiles.csv", index=False)
kernel_four_day.to_csv(TABLE_DIR / "focused_5x5_four_day_profiles.csv", index=False)
kernel_anchors.to_csv(TABLE_DIR / "focused_5x5_anchors.csv", index=False)
complete_features.reset_index().to_csv(TABLE_DIR / "focused_clustering_features.csv", index=False)
family_id.rename("family_id").reset_index().to_csv(TABLE_DIR / "focused_recovery_families.csv", index=False)
brightness_diagnostic.to_csv(TABLE_DIR / "focused_baseline_variability.csv", index=False)
clustering_features.to_csv(TABLE_DIR / "focused_all_trajectory_features.csv")
clustering_counts.to_csv(TABLE_DIR / "focused_all_trajectory_counts.csv")
complete_features_all.to_csv(TABLE_DIR / "focused_clustering_features_both_supports.csv")
eligibility.reset_index().to_csv(TABLE_DIR / "focused_clustering_eligibility.csv", index=False)
diagnostics_all.to_csv(TABLE_DIR / "focused_k_diagnostics.csv", index=False)
assignments.to_csv(TABLE_DIR / "focused_recovery_families_both_supports.csv", index=False)
family_summaries.to_csv(TABLE_DIR / "focused_family_summaries.csv", index=False)
settings = dict(signal=DNB_BAND, mqf=0, ghsl_mask=SETTLEMENT_MASK, ghsl_classes=GHSL_MASKS[SETTLEMENT_MASK],
    event_date=str(EVENT_DATE.date()), baseline_start=str(BASELINE_START.date()), baseline_end=str(PRE_EVENT_END.date()),
    analysis_start=str(ANALYSIS_START.date()), profile_end=str(PROFILE_END.date()),
    baseline_pixel_min_composites=MIN_BASELINE_OBSERVATIONS[4], spatial_completeness_pct=SPATIAL_COMPLETENESS_PCT,
    aggregation_days=AGGREGATION_DAYS, daily_cap_percentile=RQ_CLIP_PERCENTILE,
    cluster_horizon_days=CLUSTER_HORIZON_DAYS, cluster_bin_days=CLUSTER_BIN_DAYS,
    minimum_bin_composites=MIN_BIN_COMPOSITES, imputation="none", scaling="none: baseline-relative percentage",
    random_state=CLUSTER_RANDOM_STATE, clustering="separate by spatial support", study_areas=STUDY_AREAS)
(TABLE_DIR / "focused_analysis_settings.json").write_text(json.dumps(settings, indent=2))
print("Tables:", TABLE_DIR)
print("Interactive figures:", FIGURE_DIR)


Method reference: [scikit-learn silhouette analysis](https://scikit-learn.org/stable/auto_examples/cluster/plot_kmeans_silhouette_analysis.html). Silhouettes are defined only for 2 through n−1 realised labels. A selected k is a candidate for this small observed subset, not a demonstrated population optimum.

**Execution note.** The notebook structure and Python syntax were checked during refurbishment. The local source rasters and boundary attributes were inaccessible; full execution and empirical k-selection remain to be performed in the project environment. The old T50/T80/T90 module has been omitted from this minimum version to keep the requested trajectory-to-family sequence concise; this notebook does not overwrite those older metric files.
